In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print("MAPPA:", root)
    for f in files:
        print("   FÁJL:", f)

MAPPA: /kaggle/input
MAPPA: /kaggle/input/datasets
MAPPA: /kaggle/input/datasets/tamsknya
MAPPA: /kaggle/input/datasets/tamsknya/ml-latest-small
   FÁJL: movies.csv
   FÁJL: ratings.csv
   FÁJL: README.txt
   FÁJL: tags.csv
   FÁJL: links.csv


In [2]:
# ============================================================
# FILMAJÁNLÓ RENDSZER – KAGGLE-RE ÖSSZEVONT, EGYFÁJLOS VERZIÓ
# ============================================================
# Ez a notebook egyetlen fájlban tartalmazza:
# - konfigurációt
# - adatbetöltést
# - train/test splitet
# - user-based CF modellt
# - item-based CF modellt
# - értékelő metrikákat
# - a teljes futtatási pipeline-t
#
# Kaggle-kompatibilis:
# - NINCS google drive mount
# - NINCS külön .py fájl import
# - az adatokat a /kaggle/input/... mappából olvassa
# ============================================================


# =========================
# 1. Könyvtárak importálása
# =========================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


# =========================
# 2. Konfiguráció
# =========================
# Itt állíthatók a fő paraméterek.

# Kaggle dataset mappa
DATA_DIR = "/kaggle/input/datasets/tamsknya/ml-latest-small"

# Fájlnevek
RATINGS_FILE = "ratings.csv"
MOVIES_FILE = "movies.csv"

# Teszt arányok, amiket ki akarunk próbálni
TEST_SIZES = [0.30, 0.25, 0.20]   # 70/30, 75/25, 80/20

# Reprodukálhatóság
RANDOM_STATE = 42

# CF paraméterek
K_NEIGHBORS = 30
MIN_COMMON = 2

# Értékelés
RELEVANT_RATING_THRESHOLD = 4.0
TOP_K_RECOMMENDATIONS = 10

# Ha gyorsítani akarsz, beállíthatsz mintát:
# pl. 500 vagy 1000; None esetén az összes értékelhető tesztpár
TEST_SAMPLE_SIZE = None


# =========================
# 3. Adatbetöltés
# =========================
def load_data():
    """
    Betölti a MovieLens ratings és movies fájlokat Kaggle inputból.
    
    Mit csinál?
    - beolvassa a CSV-ket
    - ellenőrzi az alap oszlopokat
    - megtisztítja a rating oszlopot
    - visszaadja a ratings és movies DataFrame-eket
    """
    ratings_path = f"{DATA_DIR}/{RATINGS_FILE}"
    movies_path = f"{DATA_DIR}/{MOVIES_FILE}"

    ratings = pd.read_csv(ratings_path)
    movies = pd.read_csv(movies_path)

    required_ratings = {"userId", "movieId", "rating"}
    required_movies = {"movieId", "title", "genres"}

    if not required_ratings.issubset(ratings.columns):
        raise ValueError(f"ratings.csv-ből hiányzik: {required_ratings - set(ratings.columns)}")

    if not required_movies.issubset(movies.columns):
        raise ValueError(f"movies.csv-ből hiányzik: {required_movies - set(movies.columns)}")

    # Csak a szükséges oszlopok maradnak
    ratings = ratings[["userId", "movieId", "rating"]].copy()

    # rating numerikussá alakítása
    ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")
    ratings = ratings.dropna(subset=["rating"])

    # userId és movieId legyen egész
    ratings["userId"] = ratings["userId"].astype(int)
    ratings["movieId"] = ratings["movieId"].astype(int)

    return ratings, movies


def split_data(ratings, test_size, random_state=RANDOM_STATE):
    """
    Véletlenszerű train/test felosztás.
    
    test_size=0.30 -> 70/30
    test_size=0.25 -> 75/25
    test_size=0.20 -> 80/20
    """
    train_ratings, test_ratings = train_test_split(
        ratings,
        test_size=test_size,
        random_state=random_state
    )
    return train_ratings.reset_index(drop=True), test_ratings.reset_index(drop=True)


# =========================
# 4. User-based CF
# =========================
def _pearson_similarity(a, b, min_common=MIN_COMMON):
    """
    Pearson-korreláció két vektor között csak a közös, nem hiányzó elemek alapján.
    
    Ha túl kevés közös érték van, vagy nincs szórás, 0-t adunk vissza.
    """
    mask = ~(np.isnan(a) | np.isnan(b))
    if np.sum(mask) < min_common:
        return 0.0

    a_common = a[mask].astype(float)
    b_common = b[mask].astype(float)

    if np.std(a_common) == 0 or np.std(b_common) == 0:
        return 0.0

    return np.corrcoef(a_common, b_common)[0, 1]


def _cache_key(a, b):
    """
    Szimmetrikus cache kulcs:
    (a,b) és (b,a) ugyanaz legyen.
    """
    return (min(a, b), max(a, b))


class UserBasedCF:
    """
    Felhasználó-alapú kollaboratív szűrés.
    
    Ötlet:
    Ha két user hasonlóan értékelte a közös filmeket,
    akkor egy új filmre is hasonló értékelést adhatnak.
    """

    def __init__(self, k=K_NEIGHBORS, min_common=MIN_COMMON):
        self.k = k
        self.min_common = min_common
        self.user_item_ = None
        self.user_means_ = None
        self.movie_to_users_ = None
        self.train_users_ = None
        self.train_movies_ = None
        self._sim_cache_ = {}

    def fit(self, train_ratings):
        """
        A modell 'tanítása':
        - user-item mátrix készítése
        - userátlagok kiszámítása
        - movie -> [(user, rating)] index építése
        """
        self.user_item_ = train_ratings.pivot_table(
            index="userId", columns="movieId", values="rating"
        )

        self.user_means_ = self.user_item_.mean(axis=1)

        self.movie_to_users_ = {}
        for movie_id, group in train_ratings.groupby("movieId"):
            self.movie_to_users_[movie_id] = list(zip(group["userId"].values, group["rating"].values))

        self.train_users_ = set(self.user_item_.index)
        self.train_movies_ = set(self.user_item_.columns)
        self._sim_cache_ = {}
        return self

    def predict(self, user_id, movie_id):
        """
        Egyetlen (user, movie) párra rating előrejelzés.
        
        Logika:
        - ha új user -> globális átlag
        - ha új film -> user átlaga
        - különben:
          a filmet értékelő többi user közül megkeressük a leghasonlóbbakat,
          majd súlyozott átlaggal becsülünk.
        """
        if self.user_item_ is None:
            raise ValueError("A modell nincs betanítva. Előbb fit() kell.")

        # Ha a user nincs a trainben, globális átlag
        if user_id not in self.train_users_:
            return float(self.user_means_.mean())

        # Ha a film nincs a trainben, user átlag
        if movie_id not in self.movie_to_users_:
            return float(self.user_means_.get(user_id, self.user_means_.mean()))

        users_rated_movie = self.movie_to_users_[movie_id]
        if not users_rated_movie:
            return float(self.user_means_.get(user_id, self.user_means_.mean()))

        user_row = self.user_item_.loc[user_id]
        similarities = []
        ratings_for_movie = []

        for other_uid, other_rating in users_rated_movie:
            if other_uid == user_id:
                continue

            key = _cache_key(user_id, other_uid)

            if key not in self._sim_cache_:
                other_row = self.user_item_.loc[other_uid]
                common = user_row.index.intersection(other_row.index)

                if len(common) < self.min_common:
                    self._sim_cache_[key] = 0.0
                else:
                    r_a = user_row[common].values
                    r_b = other_row[common].values
                    self._sim_cache_[key] = _pearson_similarity(r_a, r_b, self.min_common)

            sim = self._sim_cache_[key]

            # Csak pozitív hasonlóságot használunk
            if sim > 0:
                similarities.append(sim)
                ratings_for_movie.append(other_rating)

        # Ha nincs használható szomszéd -> user átlag
        if len(similarities) == 0:
            return float(self.user_means_.get(user_id, self.user_means_.mean()))

        similarities = np.array(similarities)
        ratings_for_movie = np.array(ratings_for_movie)

        # Csak a top-k legnagyobb hasonlóságú user
        if len(similarities) > self.k:
            top_idx = np.argsort(similarities)[-self.k:]
            similarities = similarities[top_idx]
            ratings_for_movie = ratings_for_movie[top_idx]

        # Súlyozott átlag
        pred = np.dot(similarities, ratings_for_movie) / np.sum(similarities)

        # Vágjuk a rating skálára
        return float(np.clip(pred, 0.5, 5.0))

    def predict_batch(self, test_df):
        """
        Több (user, movie) párra egyszerre készít előrejelzést.
        """
        preds = []
        for row in test_df.itertuples(index=False):
            preds.append(self.predict(row.userId, row.movieId))
        return np.array(preds)


# =========================
# 5. Item-based CF
# =========================
class ItemBasedCF:
    """
    Elem-alapú kollaboratív szűrés.
    
    Ötlet:
    Ha egy user egy filmhez hasonló filmeket kedvelt,
    akkor a célfilmet is valószínűleg hasonlóan értékeli.
    """

    def __init__(self, k=K_NEIGHBORS, min_common=MIN_COMMON):
        self.k = k
        self.min_common = min_common
        self.item_user_ = None
        self.item_means_ = None
        self.user_to_movies_ = None
        self.train_users_ = None
        self.train_movies_ = None
        self._sim_cache_ = {}

    def fit(self, train_ratings):
        """
        A modell 'tanítása':
        - item-user mátrix építése
        - itemátlagok kiszámítása
        - user -> [(movie, rating)] index építése
        """
        self.item_user_ = train_ratings.pivot_table(
            index="movieId", columns="userId", values="rating"
        )

        self.item_means_ = self.item_user_.mean(axis=1)

        self.user_to_movies_ = {}
        for user_id, group in train_ratings.groupby("userId"):
            self.user_to_movies_[user_id] = list(zip(group["movieId"].values, group["rating"].values))

        self.train_users_ = set(self.item_user_.columns)
        self.train_movies_ = set(self.item_user_.index)
        self._sim_cache_ = {}
        return self

    def predict(self, user_id, movie_id):
        """
        Egyetlen (user, movie) párra rating előrejelzés.
        
        Logika:
        - ha új user -> globális itemátlag
        - ha új film -> itemátlag / globális átlag
        - különben:
          a user korábban értékelt filmjei közül keressük a célfilmhez
          leghasonlóbbakat, majd súlyozott átlaggal becsülünk.
        """
        if self.item_user_ is None:
            raise ValueError("A modell nincs betanítva. Előbb fit() kell.")

        if user_id not in self.train_users_:
            return float(self.item_means_.mean())

        if movie_id not in self.train_movies_:
            return float(self.item_means_.mean())

        user_rated_movies = self.user_to_movies_.get(user_id, [])
        if not user_rated_movies:
            return float(self.item_means_.get(movie_id, self.item_means_.mean()))

        movie_row = self.item_user_.loc[movie_id]
        similarities = []
        user_ratings = []

        for other_mid, user_rating in user_rated_movies:
            if other_mid == movie_id:
                continue
            if other_mid not in self.item_user_.index:
                continue

            key = _cache_key(movie_id, other_mid)

            if key not in self._sim_cache_:
                other_row = self.item_user_.loc[other_mid]
                common = movie_row.index.intersection(other_row.index)

                if len(common) < self.min_common:
                    self._sim_cache_[key] = 0.0
                else:
                    r_a = movie_row[common].values
                    r_b = other_row[common].values
                    self._sim_cache_[key] = _pearson_similarity(r_a, r_b, self.min_common)

            sim = self._sim_cache_[key]

            # Csak pozitív hasonlóság
            if sim > 0:
                similarities.append(sim)
                user_ratings.append(user_rating)

        if len(similarities) == 0:
            return float(self.item_means_.get(movie_id, self.item_means_.mean()))

        similarities = np.array(similarities)
        user_ratings = np.array(user_ratings)

        # Top-k hasonló item
        if len(similarities) > self.k:
            top_idx = np.argsort(similarities)[-self.k:]
            similarities = similarities[top_idx]
            user_ratings = user_ratings[top_idx]

        pred = np.dot(similarities, user_ratings) / np.sum(similarities)

        return float(np.clip(pred, 0.5, 5.0))

    def predict_batch(self, test_df):
        """
        Több (user, movie) párra előrejelzés.
        """
        preds = []
        for row in test_df.itertuples(index=False):
            preds.append(self.predict(row.userId, row.movieId))
        return np.array(preds)


# =========================
# 6. Értékelő metrikák
# =========================
def rmse(predictions, actual):
    """
    Gyöknégyzetes középhiba.
    Minél kisebb, annál jobb.
    """
    predictions = np.asarray(predictions)
    actual = np.asarray(actual)
    return float(np.sqrt(np.mean((predictions - actual) ** 2)))


def mae(predictions, actual):
    """
    Átlagos abszolút hiba.
    Minél kisebb, annál jobb.
    """
    predictions = np.asarray(predictions)
    actual = np.asarray(actual)
    return float(np.mean(np.abs(predictions - actual)))


def precision_recall_at_k(test_ratings, pred_user, pred_item, k=TOP_K_RECOMMENDATIONS,
                          relevant_threshold=RELEVANT_RATING_THRESHOLD):
    """
    Macro Precision@k és Recall@k számítása mindkét modellre.
    
    Mit csinál?
    - userenként rendezi a tesztben szereplő filmeket az előrejelzett rating szerint
    - a top-k elemet ajánlásnak tekinti
    - megnézi, ebből hány releváns (valós rating >= küszöb)
    - userenként számol, majd átlagol
    """
    test_df = test_ratings.copy()
    test_df["pred_user"] = pred_user
    test_df["pred_item"] = pred_item
    test_df["relevant"] = test_df["rating"] >= relevant_threshold

    def _one_model(pred_col):
        precisions = []
        recalls = []

        for user_id, group in test_df.groupby("userId"):
            group = group.sort_values(pred_col, ascending=False).reset_index(drop=True)
            top_k = group.head(k)

            n_relevant_in_top = top_k["relevant"].sum()
            n_relevant_total = group["relevant"].sum()

            prec = n_relevant_in_top / k if k > 0 else 0.0
            rec = n_relevant_in_top / n_relevant_total if n_relevant_total > 0 else 0.0

            precisions.append(prec)
            recalls.append(rec)

        if len(precisions) == 0:
            return 0.0, 0.0

        return float(np.mean(precisions)), float(np.mean(recalls))

    prec_user, rec_user = _one_model("pred_user")
    prec_item, rec_item = _one_model("pred_item")

    return (prec_user, rec_user), (prec_item, rec_item)


def evaluate(test_ratings, pred_user, pred_item):
    """
    Összes fontos metrika kiszámítása mindkét modellre.
    """
    actual = test_ratings["rating"].values

    (prec_user, rec_user), (prec_item, rec_item) = precision_recall_at_k(
        test_ratings, pred_user, pred_item
    )

    return {
        "rmse_user": rmse(pred_user, actual),
        "mae_user": mae(pred_user, actual),
        "rmse_item": rmse(pred_item, actual),
        "mae_item": mae(pred_item, actual),
        "precision_at_k_user": prec_user,
        "recall_at_k_user": rec_user,
        "precision_at_k_item": prec_item,
        "recall_at_k_item": rec_item,
    }


# =========================
# 7. Egy split kiértékelése
# =========================
def run_one_experiment(ratings, test_size):
    """
    Egy adott train/test felosztásra lefuttatja a teljes kísérletet:
    - split
    - cold-start szűrés
    - user-based fit + pred
    - item-based fit + pred
    - metrikák számítása
    """
    train_ratings, test_ratings = split_data(ratings, test_size=test_size)

    # Cold-start párok kiszűrése:
    # csak azokat értékeljük, ahol a user és a film is szerepelt a trainben
    train_users = set(train_ratings["userId"])
    train_movies = set(train_ratings["movieId"])

    mask = (
        test_ratings["userId"].isin(train_users) &
        test_ratings["movieId"].isin(train_movies)
    )

    test_eval = test_ratings.loc[mask].reset_index(drop=True)

    # Opcionális mintavétel gyorsításhoz
    if TEST_SAMPLE_SIZE is not None and len(test_eval) > TEST_SAMPLE_SIZE:
        test_eval = test_eval.sample(
            n=TEST_SAMPLE_SIZE,
            random_state=RANDOM_STATE
        ).reset_index(drop=True)

    # Modellek tanítása
    user_cf = UserBasedCF(k=K_NEIGHBORS, min_common=MIN_COMMON)
    user_cf.fit(train_ratings)

    item_cf = ItemBasedCF(k=K_NEIGHBORS, min_common=MIN_COMMON)
    item_cf.fit(train_ratings)

    # Előrejelzések
    pred_user = user_cf.predict_batch(test_eval)
    pred_item = item_cf.predict_batch(test_eval)

    # Metrikák
    metrics = evaluate(test_eval, pred_user, pred_item)

    return metrics


# =========================
# 8. Főprogram
# =========================
def main():
    """
    A teljes pipeline futtatása.
    
    Mit csinál?
    1. Adatok betöltése
    2. 70/30, 75/25, 80/20 kísérletek lefuttatása
    3. Eredmények összegyűjtése
    4. Két külön táblázat kiírása:
       - RMSE, MAE
       - Precision@10, Recall@10
    """
    print("Adatok betöltése...")
    ratings, movies = load_data()
    print(f"Betöltve: {len(ratings)} rating, {len(movies)} film")

    results_rows_error = []
    results_rows_topk = []

    for test_size in TEST_SIZES:
        train_pct = int((1 - test_size) * 100)
        test_pct = int(test_size * 100)

        print(f"\n===== Kísérlet: {train_pct}/{test_pct} =====")

        metrics = run_one_experiment(ratings, test_size)

        results_rows_error.append({
            "Train": f"{train_pct}/{test_pct}",
            "Módszer": "User-based",
            "RMSE": round(metrics["rmse_user"], 6),
            "MAE": round(metrics["mae_user"], 6),
        })
        results_rows_error.append({
            "Train": f"{train_pct}/{test_pct}",
            "Módszer": "Item-based",
            "RMSE": round(metrics["rmse_item"], 6),
            "MAE": round(metrics["mae_item"], 6),
        })

        results_rows_topk.append({
            "Train": f"{train_pct}/{test_pct}",
            "Módszer": "User-based",
            f"Precision@{TOP_K_RECOMMENDATIONS}": round(metrics["precision_at_k_user"], 6),
            f"Recall@{TOP_K_RECOMMENDATIONS}": round(metrics["recall_at_k_user"], 6),
        })
        results_rows_topk.append({
            "Train": f"{train_pct}/{test_pct}",
            "Módszer": "Item-based",
            f"Precision@{TOP_K_RECOMMENDATIONS}": round(metrics["precision_at_k_item"], 6),
            f"Recall@{TOP_K_RECOMMENDATIONS}": round(metrics["recall_at_k_item"], 6),
        })

    df_error = pd.DataFrame(results_rows_error)
    df_topk = pd.DataFrame(results_rows_topk)

    print("\n\n===== EREDMÉNYEK: RMSE / MAE =====")
    print(df_error.to_string(index=False))

    print("\n\n===== EREDMÉNYEK: PRECISION / RECALL =====")
    print(df_topk.to_string(index=False))

    return df_error, df_topk


# =========================
# 9. Futtatás
# =========================
# A notebookban ezzel indul el az egész kísérlet.
df_error, df_topk = main()

Adatok betöltése...
Betöltve: 100836 rating, 9742 film

===== Kísérlet: 70/30 =====

===== Kísérlet: 75/25 =====

===== Kísérlet: 80/20 =====


===== EREDMÉNYEK: RMSE / MAE =====
Train    Módszer     RMSE      MAE
70/30 User-based 0.978530 0.754748
70/30 Item-based 0.984761 0.761577
75/25 User-based 0.970426 0.749167
75/25 Item-based 0.980674 0.759564
80/20 User-based 0.972262 0.750189
80/20 Item-based 0.983170 0.762194


===== EREDMÉNYEK: PRECISION / RECALL =====
Train    Módszer  Precision@10  Recall@10
70/30 User-based      0.618033   0.581367
70/30 Item-based      0.540000   0.545085
75/25 User-based      0.592131   0.622212
75/25 Item-based      0.520164   0.584510
80/20 User-based      0.555410   0.674679
80/20 Item-based      0.491311   0.639720
